# FreePhish: classify the same URLs with all four saved models (Colab)
Run the cells **top to bottom**.

This is a small, standalone demo notebook, separate from the full training pipeline. It assumes you
already have the four trained model files from earlier runs (upload whichever ones you have):

| File | Produced by |
|---|---|
| `model.joblib` | `freephish_old.py train` (baseline, 22 features) |
| `model_enh.joblib` | `freephish_enhanced.py train` (enhanced, 100 features) |
| `model_adv.joblib` | `freephish_gan.py tune --save-model ...` (adversarially-trained enhanced) |
| `model_de.joblib` | `freephish_de.py train` (DE feature-weighted enhanced) |

You don't need all four - the classify-all script accepts any subset and just skips the missing ones.

**Flow:** upload scripts + models -> build one shared `urls.txt` (live phishing + benign pages) ->
**live monitoring across all four models, at the end**, on that same URL list.

## 0. Upload the scripts

In [ ]:
%cd /content
!rm -f freephish_*.py requirements.txt
from google.colab import files
files.upload()   # select: freephish_old.py, freephish_enhanced.py, freephish_gan.py, freephish_de.py, freephish_classify_all.py, requirements.txt
!ls

## 1. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 2. Upload your already-trained models
Select whichever of `model.joblib` / `model_enh.joblib` / `model_adv.joblib` / `model_de.joblib` you
already have from earlier sessions. If you're missing one, just skip it here - the next cell detects
which files actually arrived and only passes those to the comparison at the end.

In [ ]:
from google.colab import files
uploaded = files.upload()   # select any/all of: model.joblib, model_enh.joblib, model_adv.joblib, model_de.joblib
!ls -la model*.joblib 2>/dev/null || echo "(no model files uploaded yet)" 

### 2b. Don't have a saved adversarial model yet?
`freephish_gan.py tune` (and `retrain`) didn't used to save the adversarially-trained model to disk -
it only wrote a metrics JSON. The copy of `freephish_gan.py` you uploaded in Step 0 adds a `--save-model`
flag that fixes this. Run this cell **only if you need to produce `model_adv.joblib` right now** (needs
`model_enh.joblib`, `sample.parquet`, and `results_enhanced.predictions.csv` from an earlier Stage-2 run -
upload those too if they're missing); otherwise skip straight to Section 3.

In [ ]:
# Optional - only run this if you don't already have model_adv.joblib
!python freephish_gan.py tune --csv sample.parquet \
  --enhanced-model model_enh.joblib --enhanced-results-pred results_enhanced.predictions.csv \
  --epsilon-grid 0.5 1.0 1.5 --epochs-grid 100 150 \
  --save-model model_adv.joblib --out tune_results.json

## 3. Build `urls.txt`: live PhishTank (phishing) + GitHub Pages (benign)
Same live-fetch cell as the full pipeline notebook: up to `N` phishing URLs from PhishTank's feed
(falling back to the Phishing.Database GitHub mirror if PhishTank is unreachable) and up to `N` live,
reachable GitHub Pages sites, both filtered to the free-web-builder (FWB) domains the models were trained
on. The final URL count is `2 * min(N, phishing_found, benign_found)` - **if either side comes back short,
the total is smaller than `2*N`,** that's expected live-API variance, not a bug.

In [ ]:
import os, io, requests, pandas as pd
from urllib.parse import urlparse

FWB_SUFFIXES = [
    "weebly.com", "000webhostapp.com", "blogspot.com", "wixsite.com", "wix.com",
    "sites.google.com", "github.io", "firebaseapp.com", "web.app", "square.site",
    "wordpress.com", "forms.gle", "docs.google.com", "sharepoint.com", "yolasite.com",
    "godaddysites.com", "mailchimpsites.com", "glitch.me", "hpage.com", "zohopublic.com",
]
def is_fwb(url):
    host = (urlparse(url).hostname or "").lower()
    return any(host == s or host.endswith("." + s) for s in FWB_SUFFIXES)

N = 10
UA = {"User-Agent": "phishtank/research-demo"}

# ---- 1) PHISHING: PhishTank verified+online feed (optional app key in env PHISHTANK_KEY) ----
key = os.environ.get("PHISHTANK_KEY")
pt_url = (f"http://data.phishtank.com/data/{key}/online-valid.csv" if key
          else "http://data.phishtank.com/data/online-valid.csv")
phish_urls = []
try:
    r = requests.get(pt_url, headers=UA, timeout=60)
    r.raise_for_status()
    phish_urls = pd.read_csv(io.StringIO(r.text))["url"].dropna().astype(str).tolist()
    src = "PhishTank"
except Exception as e:
    print("PhishTank failed:", type(e).__name__, e, "-> falling back to Phishing.Database (GitHub)")
    r = requests.get("https://raw.githubusercontent.com/mitchellkrogza/Phishing.Database/master/phishing-links-ACTIVE.txt",
                     timeout=60)
    phish_urls = [u.strip() for u in r.text.splitlines() if u.strip()]
    src = "Phishing.Database"
fwb_phish = [u for u in phish_urls if is_fwb(u)]
print(f"{src}: {len(phish_urls)} URLs, {len(fwb_phish)} on FWB domains")

# ---- 2) BENIGN: popular public GitHub repos that have a GitHub Pages site ----
benign_urls = []
try:
    for page in (1, 2):
        j = requests.get("https://api.github.com/search/repositories",
                         params={"q": "stars:>3000", "sort": "stars", "order": "desc", "per_page": 100, "page": page},
                         headers={"Accept": "application/vnd.github+json"}, timeout=20).json()
        for repo in j.get("items", []):
            if not repo.get("has_pages"):
                continue
            u = f"https://{repo['owner']['login'].lower()}.github.io/{repo['name']}/"
            if u in benign_urls or not is_fwb(u):
                continue
            try:
                rr = requests.get(u, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
                if rr.status_code < 400 and is_fwb(rr.url):
                    benign_urls.append(u)
            except Exception:
                pass
            if len(benign_urls) >= N: break
        if len(benign_urls) >= N: break
except Exception as e:
    print("GitHub API failed:", type(e).__name__, e)
print(f"benign FWB pages found: {len(benign_urls)}")

# ---- 3) balance and write ----
n = min(N, len(fwb_phish), len(benign_urls))
urls_to_watch = fwb_phish[:n] + benign_urls[:n]
print(f"{n} phishing + {n} benign = {len(urls_to_watch)} URLs")
with open("urls.txt", "w") as f:
    f.write("\n".join(urls_to_watch))
print(f"wrote {len(urls_to_watch)} URLs to urls.txt")

## 4. Live monitoring: classify these same URLs with all four models (last step)
`freephish_classify_all.py` fetches each URL's HTML **once**, builds one shared raw feature matrix, and
then has every model you uploaded score it with its own training-time logic - the exact
`predict_original`/`predict_enhanced`/`predict_adv`/`predict_de` helpers already used inside
`freephish_gan.py`, so this is pure inference: nothing here is retrained. Only the models you actually
uploaded in Section 2 are passed in; a missing file just drops that column from the table.

In [ ]:
import os

flags = []
if os.path.exists("model.joblib"):      flags += ["--old-model", "model.joblib"]
if os.path.exists("model_enh.joblib"):  flags += ["--enhanced-model", "model_enh.joblib"]
if os.path.exists("model_adv.joblib"):  flags += ["--adv-model", "model_adv.joblib"]
if os.path.exists("model_de.joblib"):   flags += ["--de-model", "model_de.joblib"]
print("using:", flags or "(none found - upload at least one model file in Section 2)")

!python freephish_classify_all.py --urls urls.txt {' '.join(flags)} --out classify_all_results.jsonl

## 5. Inspect the saved results as a table

In [ ]:
import json, pandas as pd

rows = [json.loads(l) for l in open("classify_all_results.jsonl")]
model_cols = [k for k in rows[0] if k not in ("url", "html_fetched")]
table = pd.DataFrame([
    {"url": r["url"], "html_fetched": r["html_fetched"],
     **{m: f"{'PHISH' if r[m]['phishing'] else 'ok'} ({r[m]['p_phish']:.3f})" for m in model_cols}}
    for r in rows
])
table

## Reading the results honestly
- This is **inference only**, on models already trained and saved in earlier sessions - nothing is
  retrained here, so a disagreement between models reflects what each one actually learned, not a fresh
  fit on these specific URLs.
- A URL where `html_fetched` is `False` was classified from URL-only features (and whatever behavior
  features default to when the page can't be reached) - treat its prediction accordingly.
- Live phishing URLs get taken down quickly; by the time you run this, some of PhishTank's listed pages
  may already be dead. That's expected and is itself part of the demo story (see the full pipeline
  notebook's `analyze`/liveness step if you want to show that explicitly).